# biosec-toolkit

Codon Optimization and ATTATA Promoter Motif Injection for DNA Sequence Engineering

[GitHub](https://github.com/r1gron9/biosec-toolkit) | [Documentation](https://github.com/r1gron9/biosec-toolkit)

---

**Welcome to biosec-toolkit Google Colab!**

- Select `File` → `Save a copy in Drive` to save this notebook
- You can run each cell by clicking the ▶️ icon
- Use these sections for DNA sequence optimization with ATTATA motif injection

# Setup Notebook

## Install the Package

In [ ]:
import subprocess
import sys
import os

print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'biopython', 'pandas', 'tqdm', 'numpy'], check=True)

print('✓ Dependencies installed successfully')
print('\nNext steps:')
print('1. Run the next cell to define the core modules')
print('2. Either upload files OR use the built-in test data generator')

In [ ]:
import pandas as pd
import random
import warnings
from Bio.Seq import Seq
from Bio.Data import CodonTable
from itertools import product
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ============================================================================
# BIOSEC-TOOLKIT MODULES (Colab Edition)
# ============================================================================

# Load standard codon table
codon_table = CodonTable.unambiguous_dna_by_id[1]

# Prepare synonymous codon dictionary
synonymous_codons = {}
for codon, aa in codon_table.forward_table.items():
    synonymous_codons.setdefault(aa, []).append(codon)

# ---- generate_test_file module ----
def random_codon_for_aa(aa, exclude=None):
    options = [c for c in synonymous_codons[aa] if c != exclude]
    return random.choice(options) if options else exclude

def generate_test_file(total_sequences, output_path):
    test_entries = []
    for _ in range(total_sequences):
        expected_case_list = []
        num_cases = random.randint(1, 3)
        dna_seq = ""
        for _ in range(num_cases):
            case_num = random.choice([1, 2, 3])
            if case_num == 1:
                c1 = random_codon_for_aa('I', exclude='ATT')
                c2 = random_codon_for_aa('I', exclude='ATA')
                dna_seq += c1 + c2
            elif case_num == 2:
                c1 = random_codon_for_aa('A', exclude='GCA')
                c2 = 'TTA'
                c3 = random_codon_for_aa('Y', exclude='TAC')
                dna_seq += c1 + c2 + c3
            else:
                c1 = random_codon_for_aa('N', exclude='AAT')
                c2 = 'TAT'
                c3 = random_codon_for_aa('M', exclude='ATG')
                dna_seq += c1 + c2 + c3
        for _ in range(random.randint(0, 2)):
            dna_seq += random_codon_for_aa('S')
        if len(dna_seq) % 3 != 0:
            dna_seq += 'A' * (3 - len(dna_seq) % 3)
        protein_seq = str(Seq(dna_seq).translate())
        test_entries.append({
            "predicted_dna": dna_seq,
            "protein_sequence": protein_seq
        })
    df = pd.DataFrame(test_entries)
    df.to_csv(output_path, index=False)
    return output_path

# ---- inject_motif module ----
def insert_attata_motif(dna_seq, protein_seq):
    seq_length = len(dna_seq)
    modified_seq = list(dna_seq)
    i = 0
    motif = "ATTATA"
    total_insertions = 0
    all_changes = []
    insertion_cases = []

    while i <= seq_length - 6:
        offset = i % 3
        inserted = False

        # Case 1: 'ATT' 'ATA'
        if offset == 0:
            codon1 = dna_seq[i:i + 3]
            codon2 = dna_seq[i + 3:i + 6]
            if len(codon1) == 3 and len(codon2) == 3:
                if str(Seq(codon1).translate()) == 'I' and str(Seq(codon2).translate()) == 'I':
                    for c1_new, c2_new in product(synonymous_codons['I'], repeat=2):
                        if c1_new + c2_new == motif:
                            for j in range(3):
                                modified_seq[i + j] = c1_new[j]
                            for j in range(3):
                                modified_seq[i + 3 + j] = c2_new[j]
                            total_insertions += 1
                            insertion_cases.append(1)
                            i += 6
                            inserted = True
                            break

        if not inserted and offset == 2:
            codon1_start = i - 2
            codon2_start = i + 1
            codon3_start = i + 4
            if 0 <= codon1_start < seq_length - 2 and 0 <= codon3_start < seq_length - 2:
                codon1 = dna_seq[codon1_start:codon1_start + 3]
                codon2 = dna_seq[codon2_start:codon2_start + 3]
                codon3 = dna_seq[codon3_start:codon3_start + 3]
                aa1 = str(Seq(codon1).translate())
                aa2 = str(Seq(codon2).translate())
                aa3 = str(Seq(codon3).translate())
                if '*' not in (aa1, aa2, aa3):
                    c1_opts = [c for c in synonymous_codons.get(aa1, []) if c.endswith('A')]
                    c2_opts = [c for c in synonymous_codons.get(aa2, []) if c == 'TTA']
                    c3_opts = [c for c in synonymous_codons.get(aa3, []) if c.startswith('TA')]
                    for c1_new, c2_new, c3_new in product(c1_opts, c2_opts, c3_opts):
                        if c1_new[-1] + c2_new + c3_new[:2] == motif:
                            for j in range(3):
                                modified_seq[codon1_start + j] = c1_new[j]
                            for j in range(3):
                                modified_seq[codon2_start + j] = c2_new[j]
                            for j in range(3):
                                modified_seq[codon3_start + j] = c3_new[j]
                            total_insertions += 1
                            insertion_cases.append(2)
                            i += 6
                            inserted = True
                            break

        if not inserted and offset == 1:
            codon1_start = i - 1
            codon2_start = i + 2
            codon3_start = i + 5
            if 0 <= codon1_start < seq_length - 2 and 0 <= codon3_start < seq_length - 2:
                codon1 = dna_seq[codon1_start:codon1_start + 3]
                codon2 = dna_seq[codon2_start:codon2_start + 3]
                codon3 = dna_seq[codon3_start:codon3_start + 3]
                aa1 = str(Seq(codon1).translate())
                aa2 = str(Seq(codon2).translate())
                aa3 = str(Seq(codon3).translate())
                if '*' not in (aa1, aa2, aa3):
                    c1_opts = [c for c in synonymous_codons.get(aa1, []) if c.endswith('AT')]
                    c2_opts = [c for c in synonymous_codons.get(aa2, []) if c == 'TAT']
                    c3_opts = [c for c in synonymous_codons.get(aa3, []) if c.startswith('A')]
                    for c1_new, c2_new, c3_new in product(c1_opts, c2_opts, c3_opts):
                        if c1_new[-2:] + c2_new + c3_new[0] == motif:
                            for j in range(3):
                                modified_seq[codon1_start + j] = c1_new[j]
                            for j in range(3):
                                modified_seq[codon2_start + j] = c2_new[j]
                            for j in range(3):
                                modified_seq[codon3_start + j] = c3_new[j]
                            total_insertions += 1
                            insertion_cases.append(3)
                            i += 6
                            inserted = True
                            break
        if not inserted:
            i += 1

    return ''.join(modified_seq), total_insertions, all_changes, insertion_cases

def process_csv(input_path, output_filename):
    df = pd.read_csv(input_path, dtype={0: str, 2: str})
    results = []
    inserted_count = 0
    total_sequences = len(df)

    for _, row in tqdm(df.iterrows(), total=total_sequences, desc="Processing"):
        dna_seq = row["predicted_dna"].upper()
        protein_seq = row["protein_sequence"].upper()
        if "ATTATA" in dna_seq:
            result = {
                "Original DNA": dna_seq,
                "Modified DNA": dna_seq,
                "Num Insertions": 0,
                "Existing Motif": True
            }
        else:
            mod_seq, num_ins, changes, cases = insert_attata_motif(dna_seq, protein_seq)
            if num_ins > 0:
                inserted_count += 1
            result = {
                "Original DNA": dna_seq,
                "Modified DNA": mod_seq,
                "Num Insertions": num_ins,
                "Existing Motif": False
            }
        results.append(result)

    df_results = pd.DataFrame(results)
    df_results.to_csv(output_filename, index=False)
    return output_filename

# ---- search_motifs_quantity module ----
def search_motif(csv_path, motif):
    df = pd.read_csv(csv_path)
    stats = {
        'Total Sequences': len(df),
        'Sequences with Motif': 0,
        'Total Motif Count': 0,
        'Average Motifs per Sequence': 0.0,
    }
    
    for _, row in df.iterrows():
        seq = str(row.iloc[1]).upper()
        count = seq.count(motif)
        if count > 0:
            stats['Sequences with Motif'] += 1
            stats['Total Motif Count'] += count
    
    if stats['Sequences with Motif'] > 0:
        stats['Average Motifs per Sequence'] = stats['Total Motif Count'] / stats['Sequences with Motif']
    
    return stats

# ---- compare_sequences module ----
def compare_seq(seq1, seq2):
    seq1, seq2 = str(seq1).upper(), str(seq2).upper()
    min_len = min(len(seq1), len(seq2))
    differences = sum(1 for i in range(min_len) if seq1[i] != seq2[i])
    differences += abs(len(seq1) - len(seq2))
    return {
        "differences": differences,
        "seq1_length": len(seq1),
        "seq2_length": len(seq2)
    }

print('✓ All modules loaded successfully')


## Import the Package

In [ ]:
print('✓ All core modules are now available:')
print('  - generate_test_file()')
print('  - inject_motif via process_csv()')
print('  - search_motif()')
print('  - compare_seq()')
print('\nReady to process DNA sequences!')

# Optimizing DNA Sequences

## Generate Test Sequences

Generate synthetic DNA sequences for testing and optimization

In [ ]:
# Generate synthetic test sequences
num_sequences = 10  # Change this value for more/fewer sequences

print(f'Generating {num_sequences} test sequences...')
generate_test_file(total_sequences=num_sequences, output_path='test_sequences.csv')

df = pd.read_csv('test_sequences.csv')
print(f'✓ Generated {len(df)} sequences')
print(f'\nSample sequences:')
print(df.head(3))

## Inject ATTATA Motif

Inject ATTATA promoter motifs into sequences while preserving the encoded protein

In [ ]:
# Inject ATTATA motif into the generated sequences
print('Injecting ATTATA motifs...')
process_csv('test_sequences.csv', 'optimized_sequences.csv')

df_optimized = pd.read_csv('optimized_sequences.csv')
print(f'✓ Motif injection complete')
print(f'Processed: {len(df_optimized)} sequences')
print(f'\nOptimized sequences:')
print(df_optimized.head(3))

## Analyze Results

Analyze motif distribution and sequence statistics in the optimized sequences

In [ ]:
# Analyze the motif distribution
print('Analyzing motif statistics...')
stats = search_motif('optimized_sequences.csv', 'ATTATA')

print('✓ Analysis complete\n')
print('ATTATA Motif Statistics:')
print('─' * 50)
for key, value in stats.items():
    print(f'{key:.<40} {value}')

## Compare Sequences

Compare original and optimized sequences to visualize the differences

In [ ]:
# Compare the first original and optimized sequences
if len(df) > 0 and len(df_optimized) > 0:
    original_seq = df.iloc[0]['predicted_dna']
    optimized_seq = df_optimized.iloc[0]['Modified DNA']
    
    result = compare_seq(original_seq, optimized_seq)
    
    print('Sequence Comparison (First Sequence):')
    print('─' * 50)
    print(f'Original length:   {result["seq1_length"]} bp')
    print(f'Optimized length:  {result["seq2_length"]} bp')
    print(f'Differences:       {result["differences"]} positions')
    print(f'\nOriginal:  {original_seq[:60]}...')
    print(f'Optimized: {optimized_seq[:60]}...')
else:
    print('No sequences to compare')

## Download Results

Files are ready to download from the Files panel on the left side

In [ ]:
# List all generated CSV files
csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]

if csv_files:
    print('Generated files ready for download:')
    print('─' * 50)
    for f in csv_files:
        size = os.path.getsize(f) / 1024  # KB
        print(f'  ✓ {f:.<35} {size:.1f} KB')
    
    print('\n📥 Download files:')
    print('  Right-click file in left panel and select "Download"')
    print('  Or use: files.download("filename.csv")')
else:
    print('No CSV files generated yet. Run cells above first.')